# Two POVs + Trainable Feature Extractor with Regression Head

In [1]:
import math
import datetime
from sklearn.metrics import mean_squared_error, r2_score
from itertools import product
from tqdm import tqdm
from src.constants import *
from src.helpers import *
from src.trainable_pipeline import *

seed_everything(RANDOM_STATE)

## Load Data

In [2]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
print("Samples shape:", samples.shape)
display(samples[["exp_id", "volume", "top_path", "side_path"]].head())

Samples shape: (420, 24)


,exp_id,volume,top_path,side_path
0,1,1.06,photos\top_view_images\P2260331.JPG,photos\side_view_images\P2260685.JPG
1,1,2.12,photos\top_view_images\P2260332.JPG,photos\side_view_images\P2260686.JPG
2,1,3.18,photos\top_view_images\P2260333.JPG,photos\side_view_images\P2260687.JPG
3,1,4.24,photos\top_view_images\P2260334.JPG,photos\side_view_images\P2260688.JPG
4,2,1.06,photos\top_view_images\P2260335.JPG,photos\side_view_images\P2260689.JPG


In [3]:
def evaluate_setting_nested_cv(samples_df, backbone_name, fusion_name, head_name, mode_name):
    y = samples_df["volume"].to_numpy(dtype=float)
    groups = samples_df["exp_id"].to_numpy()

    outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
    outer_splits = list(outer_cv.split(np.zeros((len(samples_df), 1)), y, groups))
    fold_records = []
    oof_pred = np.full(len(samples_df), np.nan, dtype=float)

    print("=" * 90)
    print(backbone_name, fusion_name, head_name, mode_name)
    print("=" * 90)

    for fold_idx, (train_idx, test_idx) in enumerate(
        tqdm(outer_splits,
            total=len(outer_splits),
            desc=f"{backbone_name} | {fusion_name} | {head_name} | {mode_name}",),
        start=1,
    ):
        train_groups = groups[train_idx]
        n_inner = min(INNER_SPLITS, len(np.unique(train_groups)))
        inner_cv = GroupKFold(n_splits=n_inner)

        best_cfg = None
        best_inner_mae = float("inf")

        for cfg in TRAINING_CONFIGS[mode_name]:
            inner_maes = []
            for inner_train_rel, inner_val_rel in inner_cv.split(np.zeros((len(train_idx), 1)), y[train_idx], train_groups):
                inner_train_idx = train_idx[inner_train_rel]
                inner_val_idx = train_idx[inner_val_rel]
                backbone, head, preprocess, inner_val_mae = fit_model(
                    samples_df=samples_df,
                    train_idx=inner_train_idx,
                    val_idx=inner_val_idx,
                    backbone_name=backbone_name,
                    fusion_name=fusion_name,
                    head_name=head_name,
                    mode_name=mode_name,
                    cfg=cfg,
                    head_cfg=HEAD_CONFIGS[head_name],
                    batch_size=BATCH_SIZE,
                    max_epochs=MAX_EPOCHS,
                    seed=RANDOM_STATE + fold_idx,
                    device=DEVICE,
                    side_mask_paths=SIDE_ROI_MASKS,
                    top_mask_paths=TOP_ROI_MASKS,
                )
                inner_maes.append(inner_val_mae)
                del backbone, head
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            mean_inner_mae = float(np.mean(inner_maes))
            if mean_inner_mae < best_inner_mae:
                best_inner_mae = mean_inner_mae
                best_cfg = cfg

        final_train_idx, final_val_idx = make_group_train_val_split(train_idx, groups, seed=RANDOM_STATE + fold_idx)
        backbone, head, preprocess, _ = fit_model(
            samples_df=samples_df,
            train_idx=final_train_idx,
            val_idx=final_val_idx,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
            head_name=head_name,
            mode_name=mode_name,
            cfg=best_cfg,
            head_cfg=HEAD_CONFIGS[head_name],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            seed=RANDOM_STATE + fold_idx,
            device=DEVICE,
            side_mask_paths=SIDE_ROI_MASKS,
            top_mask_paths=TOP_ROI_MASKS,
        )

        y_test_true, y_test_pred = predict_indices(samples_df, test_idx, preprocess, backbone_name, backbone, head, fusion_name, BATCH_SIZE, DEVICE)
        oof_pred[test_idx] = y_test_pred

        record = {
            "fold": fold_idx,
            "backbone": backbone_name,
            "fusion": fusion_name,
            "head": head_name,
            "mode": mode_name,
            "inner_MAE": best_inner_mae,
            "MAE": mean_absolute_error(y_test_true, y_test_pred),
            "MSE": mean_squared_error(y_test_true, y_test_pred),
            "RMSE": math.sqrt(mean_squared_error(y_test_true, y_test_pred)),
            "R2": r2_score(y_test_true, y_test_pred) if len(np.unique(y_test_true)) > 1 else np.nan,
            "best_cfg": best_cfg,
        }
        fold_records.append(record)

        print(f"Fold {fold_idx}: MAE={record['MAE']:.3f}, RMSE={record['RMSE']:.3f}, R2={record['R2']}, best={best_cfg}")

        del backbone, head
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return fold_records, oof_pred

def summarise_records(records):
    rows = []
    grouped = {}
    for rec in records:
        key = (rec["backbone"], rec["fusion"], rec["head"], rec["mode"])
        grouped.setdefault(key, []).append(rec)

    for (backbone, fusion, head, mode), folds in grouped.items():
        rows.append({
            "backbone": backbone,
            "fusion": fusion,
            "head": head,
            "mode": mode,
            "cv_mae_mean": np.mean([f["MAE"] for f in folds]),
            "cv_mae_std": np.std([f["MAE"] for f in folds]),
            "cv_rmse_mean": np.mean([f["RMSE"] for f in folds]),
            "cv_rmse_std": np.std([f["RMSE"] for f in folds]),
            "cv_r2_mean": np.mean([f["R2"] for f in folds]),
            "cv_r2_std": np.std([f["R2"] for f in folds]),
            "inner_mae_mean": np.mean([f["inner_MAE"] for f in folds]),
        })

    return pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)


In [4]:
all_fold_records = []
oof_store = {}

all_configs = list(product(
    BACKBONE_NAMES,
    FUSION_NAMES,
    HEAD_CONFIGS.keys(),
    TRAINING_CONFIGS.keys()
))


for backbone_name, fusion_name, head_name, mode_name in tqdm(all_configs, desc="Experiment configs"):
    print(f"\n=== {backbone_name} | {fusion_name} | {head_name} | {mode_name} ===")
    fold_records, oof_pred = evaluate_setting_nested_cv(
        samples_df=samples,
        backbone_name=backbone_name,
        fusion_name=fusion_name,
        head_name=head_name,
        mode_name=mode_name,
    )
    all_fold_records.extend(fold_records)
    oof_store[(backbone_name, fusion_name, head_name, mode_name)] = oof_pred

summary_df = summarise_records(all_fold_records)
summary_df

Experiment configs:   0%|          | 0/96 [00:00<?, ?it/s]


=== resnet50 | concat | linear | last_stage ===
resnet50 concat linear last_stage



resnet50 | concat | linear | last_stage:   0%|          | 0/5 [00:00<?, ?it/s]

Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

resnet50 | concat | linear | last_stage:   0%|          | 0/5 [00:36<?, ?it/s]
Experiment configs:   0%|          | 0/96 [00:36<?, ?it/s]

KeyboardInterrupt



In [ ]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
name = f"results_trainable_performance_{timestamp}.csv"
summary_df.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)

In [ ]:
df = pd.DataFrame(all_fold_records)
name = f"results_trainable_performance_folds_{timestamp}.csv"
df.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)